In [2]:
import os

REPO_DIR = "/kaggle/working/sam2"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/facebookresearch/sam2.git {REPO_DIR}
else:
    print("sam2 already cloned, skipping")

%cd {REPO_DIR}
!pip install -e . -q
!pip install opencv-python Pillow tqdm lxml -q

import torch
import cv2
import numpy as np
from PIL import Image
from tqdm import tqdm
import xml.etree.ElementTree as ET
from xml.dom import minidom

os.makedirs(f"{REPO_DIR}/checkpoints", exist_ok=True)
ckpt_path = f"{REPO_DIR}/checkpoints/sam2_hiera_large.pt"
if not os.path.exists(ckpt_path):
    !wget -P {REPO_DIR}/checkpoints/ https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt
else:
    print("checkpoint already present, skipping download")

from sam2.build_sam import build_sam2_video_predictor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

sam2 already cloned, skipping
/kaggle/working/sam2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for SAM-2 (pyproject.toml) ... done
checkpoint already present, skipping download
Using device: cuda


In [3]:
# yolo 
# %pip install ultralytics

# rf-detr
%pip install rfdetr -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 768.7/768.7 kB 18.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 89.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 58.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 86.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 99.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 3.5 MB/s eta 0:00:00
ERROR: 

In [4]:
!git clone https://github.com/facebookresearch/sam2.git /kaggle/working/sam2
%cd /kaggle/working/sam2
!pip install -e . -q

!mkdir -p checkpoints
!wget -O checkpoints/sam2.1_hiera_large.pt https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt

fatal: destination path '/kaggle/working/sam2' already exists and is not an empty directory.
/kaggle/working/sam2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for SAM-2 (pyproject.toml) ... done
--2026-10-04 03:40:28--  https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 99.84.118.30, 99.84.118.117, 99.84.118.67, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|99.84.118.30|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 898083611 (856M) [application/vnd.snesdev-page-table]
Saving to: ‘checkpoints/sam2.1_hiera_large.pt’

checkpoints/sam2.1_ 100%[===================>] 856.48M   329MB/s    in 2.6s    

2026-10-04 03:40:31 (329 MB/s) - ‘checkpoints/sam2.1_hiera_large.pt’ saved [89

In [5]:
!ls -lh /kaggle/working/sam2/checkpoints

total 1.7G
-rwxr-xr-x 1 root root 2.6K Oct  4 03:32 download_ckpts.sh
-rw-r--r-- 1 root root 857M Sep 29  2024 sam2.1_hiera_large.pt
-rw-r--r-- 1 root root 857M Jul 28  2024 sam2_hiera_large.pt


### Enter your video path below

In [ ]:
import os, json, cv2, numpy as np, torch

VIDEO_PATH = "/kaggle/input/datasets/neeharikajugran/sheep-detection-new/normal1.mp4"   # <-- change this
FRAMES_DIR = "/kaggle/working/frames"
SCAN_EVERY = 30        # run YOLO every N frames
YOLO_CONF = 0.3
IOU_THRESH = 0.3       # overlap above this = already tracked
MAX_SIDE = 720         # downscale long side to save GPU memory (None = keep)

os.makedirs(FRAMES_DIR, exist_ok=True)
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
n = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    if MAX_SIDE:
        h, w = frame.shape[:2]
        s = MAX_SIDE / max(h, w)
        if s < 1:
            frame = cv2.resize(frame, (int(w * s), int(h * s)))
    cv2.imwrite(f"{FRAMES_DIR}/{n:05d}.jpg", frame)
    n += 1
cap.release()
n_frames = n
print(f"{n_frames} frames at {fps:.1f} fps")

191 frames at 25.0 fps


In [9]:
import torch
from rfdetr import RFDETRLarge
from rfdetr.assets.coco_classes import COCO_CLASSES
from sam2.build_sam import build_sam2_video_predictor

device = "cuda"
AC_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

rfdetr_model = RFDETRLarge()  # downloads COCO-pretrained RF-DETR-L weights on first run

# Find "sheep"'s class index by name rather than assuming the same index as YOLO's COCO ordering
SHEEP_CLASS_ID = COCO_CLASSES.index("sheep") if "sheep" in COCO_CLASSES else [
    k for k, v in COCO_CLASSES.items() if v == "sheep"
][0]
RFDETR_CONF = 0.3

predictor = build_sam2_video_predictor(
    "configs/sam2.1/sam2.1_hiera_l.yaml",
    "/kaggle/working/sam2/checkpoints/sam2.1_hiera_large.pt",
    device=device,
)

def detect_sheep(f):
    img = cv2.imread(f"{FRAMES_DIR}/{f:05d}.jpg")
    detections = rfdetr_model.predict(img, threshold=RFDETR_CONF)
    mask = detections.class_id == SHEEP_CLASS_ID
    boxes = detections.xyxy[mask]   # already [x1, y1, x2, y2] in pixel coords
    return [b.tolist() for b in boxes]

def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0

def mask_to_xyxy(mask):
    mask = np.squeeze(mask)
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    return [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())]

with torch.inference_mode(), torch.autocast(device, dtype=AC_DTYPE):
    state = predictor.init_state(
        video_path=FRAMES_DIR, offload_video_to_cpu=True
    )

[2026-10-04 03:40:48] [INFO] rf-detr - Downloading pretrained weights for /root/.roboflow/models/rf-detr-large-2026.pth


/root/.roboflow/models/rf-detr-large-2026.pth:   0%|          | 0.00/130M [00:00<?, ?iB/s]

[2026-10-04 03:40:50] [INFO] rf-detr - MD5 validation successful for /root/.roboflow/models/rf-detr-large-2026.pth


[2026-10-04 03:40:50] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-10-04 03:40:50] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-10-04 03:40:51] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-large-2026.pth already exists with correct MD5 hash.


frame loading (JPEG): 100%|██████████| 191/191 [00:05<00:00, 33.30it/s]


In [10]:
prompts = []          # {frame, obj_id, box}
boxes_by_frame = {}   # frame -> {obj_id: [x1,y1,x2,y2] or None}
next_id = 1
last_done = -1

def rebuild_state():
    """Reset SAM2 and replay all prompts (needed to add new object IDs)."""
    predictor.reset_state(state)
    for p in prompts:
        predictor.add_new_points_or_box(
            inference_state=state,
            frame_idx=p["frame"],
            obj_id=p["obj_id"],
            box=np.array(p["box"], dtype=np.float32),
        )

def run_propagation(start, end):
    for fi, ids, logits in predictor.propagate_in_video(
        state, start_frame_idx=start, max_frame_num_to_track=end - start
    ):
        boxes_by_frame[fi] = {
            int(oid): mask_to_xyxy((logits[i] > 0.0).cpu().numpy())
            for i, oid in enumerate(ids)
        }

with torch.inference_mode(), torch.autocast(device, dtype=AC_DTYPE):
    for f in range(0, n_frames, SCAN_EVERY):
        # 1) make sure tracking results exist up to frame f
        if prompts and last_done < f:
            start = last_done if last_done >= 0 else min(p["frame"] for p in prompts)
            run_propagation(start, f)
            last_done = f

        # 2) detect sheep, keep only those not overlapping a tracked box
        dets = detect_sheep(f)
        tracked = [b for b in boxes_by_frame.get(f, {}).values() if b is not None]
        new_dets = [d for d in dets if all(iou(d, t) < IOU_THRESH for t in tracked)]

        # 3) register new sheep, rebuild SAM2 state, re-propagate up to f
        if new_dets:
            for d in new_dets:
                prompts.append({"frame": f, "obj_id": next_id, "box": d})
                print(f"Frame {f}: new sheep -> obj_id {next_id}")
                next_id += 1
            rebuild_state()
            boxes_by_frame = {}
            run_propagation(min(p["frame"] for p in prompts), f)
            last_done = f

    # 4) finish the rest of the video
    if not prompts:
        raise SystemExit("No sheep detected in any scanned frame. Try lowering YOLO_CONF.")
    run_propagation(last_done, n_frames - 1)

print(f"Done: {next_id - 1} sheep tracked across {len(boxes_by_frame)} frames")

with open("/kaggle/working/boxes_by_frame.json", "w") as fh:
    json.dump(boxes_by_frame, fh)

[2026-10-04 03:41:03] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. For full GPU throughput (e.g. ~8x on T4 via FP16 Tensor Cores), call model.inference(dtype=torch.float16).


Frame 0: new sheep -> obj_id 1
Frame 0: new sheep -> obj_id 2
Frame 0: new sheep -> obj_id 3
Frame 0: new sheep -> obj_id 4
Frame 0: new sheep -> obj_id 5
Frame 0: new sheep -> obj_id 6
Frame 0: new sheep -> obj_id 7
Frame 0: new sheep -> obj_id 8
Frame 0: new sheep -> obj_id 9
Frame 0: new sheep -> obj_id 10
Frame 0: new sheep -> obj_id 11
Frame 0: new sheep -> obj_id 12
Frame 0: new sheep -> obj_id 13
Frame 0: new sheep -> obj_id 14
Frame 0: new sheep -> obj_id 15
Frame 0: new sheep -> obj_id 16


/kaggle/working/sam2/sam2/sam2_video_predictor.py:786: UserWarning: cannot import name '_C' from 'sam2' (/kaggle/working/sam2/sam2/__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(
propagate in video: 100%|██████████| 31/31 [02:27<00:00,  4.76s/it]


Frame 30: new sheep -> obj_id 17
Frame 30: new sheep -> obj_id 18
Frame 30: new sheep -> obj_id 19


propagate in video: 100%|██████████| 31/31 [03:15<00:00,  6.32s/it]


...

Frame 150: new sheep -> obj_id 25
Frame 150: new sheep -> obj_id 26


propagate in video: 100%|██████████| 31/31 [04:17<00:00,  8.32s/it]


Frame 180: new sheep -> obj_id 27
Frame 180: new sheep -> obj_id 28
Frame 180: new sheep -> obj_id 29


propagate in video: 100%|██████████| 11/11 [01:39<00:00,  9.06s/it]

Done: 29 sheep tracked across 191 frames


In [12]:
import os, json, zipfile, cv2

# --- Scale boxes back to the ORIGINAL video resolution (needed if MAX_SIDE resized the frames) ---
cap = cv2.VideoCapture(VIDEO_PATH)
orig_w = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
orig_h = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
cap.release()
first = cv2.imread(f"{FRAMES_DIR}/00000.jpg")
sh, sw = first.shape[:2]
sx, sy = orig_w / sw, orig_h / sh

# --- Build gt.txt: frame,id,x,y,w,h,not_ignored,class_id,visibility ---
os.makedirs("/kaggle/working/mot/gt", exist_ok=True)
lines = []
for f, objs in sorted(boxes_by_frame.items(), key=lambda kv: int(kv[0])):
    for oid, b in objs.items():
        if b is None:
            continue
        x, y = b[0] * sx, b[1] * sy
        w, h = (b[2] - b[0]) * sx, (b[3] - b[1]) * sy
        lines.append(f"{int(f) + 1},{int(oid)},{x:.2f},{y:.2f},{w:.2f},{h:.2f},1,1,1")

with open("/kaggle/working/mot/gt/gt.txt", "w") as fh:
    fh.write("\n".join(lines) + "\n")
with open("/kaggle/working/mot/gt/labels.txt", "w") as fh:
    fh.write("sheep\n")

# --- Zip it ---
with zipfile.ZipFile("/kaggle/working/mot_sheep.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("/kaggle/working/mot/gt/gt.txt", "gt/gt.txt")
    z.write("/kaggle/working/mot/gt/labels.txt", "gt/labels.txt")

print(f"{len(lines)} boxes written -> /kaggle/working/mot_sheep.zip")

4774 boxes written -> /kaggle/working/mot_sheep.zip


### Find you anotated zip in working section on the right side of the notebook